# Data Preprocessing 

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [10, 7]
plt.style.use("seaborn-v0_8")

import seaborn as sns

sns.set(style="darkgrid")

from etna.transforms import MedianOutliersTransform
from etna.transforms import TimeSeriesImputerTransform
import ipywidgets as widgets
import numpy as np
import pandas as pd

from forecasting_sticker_sales.transform import convert_to_ts_df, convert_to_df
from forecasting_sticker_sales.features import get_gdp_data

## Constants

In [ ]:
PROJECT_ROOT = Path("__file__").resolve().parents[1]

DATA_DPATH = PROJECT_ROOT / "data"
assert DATA_DPATH.exists(), "Data folder is not found"

SRC_DATA_DPATH = DATA_DPATH / "src_data"
assert SRC_DATA_DPATH.exists(), "Source data folder is not found"

OUTPUT_DATA_DPATH = DATA_DPATH / "preprocessed_data"
OUTPUT_DATA_DPATH.mkdir(parents=True, exist_ok=True)

## Data Loading 

In [ ]:
src_train_fpath = SRC_DATA_DPATH / "train.csv"

train_df = pd.read_csv(src_train_fpath, parse_dates=["date"])
train_df = train_df.sort_values(by="date")

train_df.shape

In [ ]:
train_df.head()

In [ ]:
gdb_df = get_gdp_data()
gdb_df.head()

## Preprocessing 

### Imputing Missing Values

In [ ]:
train_df["segment"] = train_df["country"] + "_" + train_df["store"] + "_" + train_df["product"]

In [ ]:
train_grouped = train_df.groupby(["segment"], as_index=False)["num_sold"].mean()

train_empty_segments = train_grouped[train_grouped["num_sold"].isna()]
train_empty_segments["country"] = train_empty_segments["segment"].apply(lambda x: x.split("_")[0])
train_empty_segments["store"] = train_empty_segments["segment"].apply(lambda x: x.split("_")[1])
train_empty_segments["product"] = train_empty_segments["segment"].apply(lambda x: x.split("_")[2])

train_empty_segments

In [ ]:
imputed_train_df = train_df.copy()
imputed_train_df["num_sold"].isna().sum()

In [ ]:
missing_train_df = imputed_train_df[imputed_train_df["num_sold"].isna()]

missing_train_df = missing_train_df.groupby(
    ["country", "store", "product", "segment"],
    as_index=False,
)
missing_train_df = missing_train_df["num_sold"].mean()

missing_train_df = missing_train_df[
    ~missing_train_df["segment"].isin(train_empty_segments["segment"])
]

missing_train_df

In [ ]:
imputed_train_df["year"] = imputed_train_df["date"].dt.year

for year in imputed_train_df["year"].unique():
    # use Norway as the max of GDP amoung the countries
    target_mask = (gdb_df["year"].dt.year == year) & (gdb_df["country"] == "Norway")
    target_ratio = gdb_df[target_mask]["ratio"].to_numpy()[0]

    for _row_idx, row in missing_train_df.iterrows():
        data_to_fill = imputed_train_df[
            (imputed_train_df["year"] == year) & imputed_train_df["country"] == row["country"]
        ]
        current_ratio = data_to_fill["ratio"].to_numpy()[0]

        norm_ratio = current_ratio / target_ratio

        mask_to_fill = (imputed_train_df["segment"] == row["segment"]) & (
            imputed_train_df["year"] == year
        )
        target_mask = (
            (imputed_train_df["country"] == "Norway")
            & (imputed_train_df["store"] == row["store"])
            & (imputed_train_df["product"] == row["product"])
            & (imputed_train_df["year"] == year)
        )

        imputed_train_df.loc[mask_to_fill, "num_sold"] = (
            imputed_train_df.loc[target_mask, "num_sold"] * norm_ratio
        ).tolist()

In [ ]:
filled_segments = []

for _row_idx, row in train_empty_segments.iterrows():
    similar_data = train_df[
        (train_df["store"] == row["store"])
        & (train_df["product"] == row["product"])
        & (train_df["country"] != row["country"])
    ]

    similar_grouped = similar_data.groupby(["date"], as_index=False)["num_sold"].mean()
    similar_grouped["num_sold"] = np.ceil(similar_grouped["num_sold"])

    train_idx = train_df[
        (train_df["country"] == row["country"])
        & (train_df["store"] == row["store"])
        & (train_df["product"] == row["product"])
    ].index

    train_df.loc[train_idx, "num_sold"] = similar_grouped["num_sold"].tolist()

In [ ]:
@widgets.interact(
    country=widgets.Dropdown(options=train_df["country"].unique()),
    store=widgets.Dropdown(options=train_empty_segments["store"].unique()),
    products=widgets.Dropdown(options=train_empty_segments["product"].unique()),
)
def show_empty_similar_data(country: str, store: str, products: str):
    plot_df = train_df[
        (train_df["country"] == country)
        & (train_df["store"] == store)
        & (train_df["product"] == products)
    ]

    fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(15, 4))

    axs.plot(plot_df["date"], plot_df["num_sold"])
    axs.set_xlabel("Date")
    axs.set_ylabel("Number of sold stickers")

    plt.show()

### Outliers, Missing Values

In [ ]:
ts_train_df = convert_to_ts_df(train_df)
print(f"Source data shape: {ts_train_df.df.shape}")

imputer = TimeSeriesImputerTransform(in_column="target", strategy="running_mean", window=30)
ts_train_df.fit_transform([imputer])
print(f"Imputer Data Shape: {ts_train_df.df.shape}")

outliers_remover = MedianOutliersTransform(in_column="target", window_size=30)
ts_train_df.fit_transform([outliers_remover])
print("Number of series with outliers:", len(outliers_remover.outliers_timestamps))
outliers_num = sum([len(values) for values in outliers_remover.outliers_timestamps.to_numpy()])
print(f"Total number of outliers: {outliers_num}")

preprocessed_train_df = convert_to_df(ts_train_df)
print(f"Missing Values: {preprocessed_train_df['num_sold'].isna().sum()}")
preprocessed_train_df = preprocessed_train_df.dropna(subset=["num_sold"])

preprocessed_train_df.shape

In [ ]:
# check for correct null filling - must be 1 days for every train_df
df_check = preprocessed_train_df.copy()
df_check["date_shifted"] = df_check["date"].shift()
df_check = df_check[~df_check["date_shifted"].isna()]
print((df_check["date"] - df_check["date_shifted"]).max())

In [ ]:
preprocessed_train_df.head()

## Data Caching

In [ ]:
train_fpath = OUTPUT_DATA_DPATH / "train.csv"
preprocessed_train_df.to_csv(train_fpath)